# 🎬 MONARCH STUDIO · WF-06 — MiniMax-H3 video lab (Colab)

**Kya hai:** MiniMax-H3 ek omni-modal video model hai — text / image / video / audio in → **video + native 32 kHz stereo audio** out, 4–15 s, 24 fps, 768P ya 2K.

**Is notebook ka kaam:** Colab ko *control room* banao. Asli generation hosted API par hoti hai
(H3-Base yahan chalane ke liye ~100–128 GB VRAM chahiye — Colab par mumkin nahi), aur Colab
prompt banata hai, jobs queue karta hai, results Drive me rakhta hai, aur ffmpeg se assemble karta hai.

| Route | Colab par? | Output | Kharcha |
|---|---|---|---|
| **A · Hosted API** (yeh notebook) | ✅ free T4/CPU par bhi | 768P + **2K**, native audio, Ref2VA | $0.04–0.13 / second |
| B · Local pruned INT8 (ComfyUI) | ⚠️ sirf Pro A100 80GB | 768P only | compute units |
| C · Local BF16 (SGLang/diffusers) | ❌ | 768P only | 4×H100 |

**Chalane se pehle:**
1. Runtime → **Change runtime type → CPU** (API route me GPU ki zaroorat nahi).
2. Left sidebar 🔑 **Secrets** → `MINIMAX_API_KEY` = aapki key, *Notebook access* ON.
3. Cells **top se neeche** chalao. Cell 8 ka `DRY_RUN` gate pehle estimate dikhayega — tab tak paisa nahi lagega.

Repo guide: `styles/minimax-h3-video/knowledge/01_COLAB_RUNBOOK.md` · prompt grammar: `02_PROMPTING.md`.

In [ ]:
#@title 1 · Environment check (no GPU needed for the API route)
import sys, os, platform, shutil, subprocess, json

print("Python :", sys.version.split()[0])
print("OS     :", platform.platform())

try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
    print("GPU    :", out if out else "none visible")
    if out:
        print("Note   : GPU sirf Route B ke liye chahiye. API route me GPU units zaya na karo —"
              " runtime CPU par rakho.")
except FileNotFoundError:
    print("GPU    : nvidia-smi nahi mila (CPU runtime) — API route ke liye bilkul theek")

print("ffmpeg :", shutil.which("ffmpeg") or "missing (Cell 11 assemble ke liye chahiye: apt-get install -y ffmpeg)")
print("RAM    :", round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30, 1), "GiB")
print("Disk   :", round(shutil.disk_usage('/').free / 2**30, 1), "GiB free")

IN_COLAB = "google.colab" in sys.modules
print("Colab  :", IN_COLAB)

In [ ]:
#@title 2 · Install, API key, config
!pip -q install requests pillow
import os, json, time, base64, mimetypes, hashlib, getpass, pathlib, subprocess
import requests

API_BASE = "https://api.minimax.io"          # CN account: https://api.minimaxi.com
CREATE   = f"{API_BASE}/v2/video_generation"
QUERY    = f"{API_BASE}/v2/query/video_generation/{{task_id}}"

def load_key():
    for name in ("MINIMAX_API_KEY", "MINIMAX_KEY"):
        try:
            from google.colab import userdata
            v = userdata.get(name)
            if v: return v.strip()
        except Exception:
            pass
        if os.environ.get(name): return os.environ[name].strip()
    return getpass.getpass("MiniMax API key: ").strip()

API_KEY = load_key()
assert API_KEY, "API key nahi mili — Colab Secrets me MINIMAX_API_KEY add karo"
HEADERS = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}

# Official list prices (USD per output second) — verify at platform.minimax.io/docs/guides/pricing-video
RATES = {
    "MiniMax-H3":           {"768P": 0.08, "2K": 0.13},
    "MiniMax-H3-Highspeed": {"768P": 0.06},
    "MiniMax-H3-Lite":      {"480P": 0.04, "768P": 0.04},
    "MiniMax-H3-Max":       {"480P": 0.05, "768P": 0.08},
}
REGEN_2K_RATE = 0.05   # 768P -> 2K regeneration, per output second

RUN_ID   = time.strftime("h3-%Y%m%d-%H%M%S")
OUT_DIR  = pathlib.Path("/content/h3_runs") / RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Run ID :", RUN_ID)
print("Output :", OUT_DIR)
print("Key    :", API_KEY[:4] + "…" + API_KEY[-3:])

## 3 · Prompt grammar (bas yahi yaad rakho)

**T2VA** — teen fields: `integrated_multimodal_description` · `overall_soundscape` · `non_diegetic_music`
**I2VA/FL2VA** — pehli line fixed instruction (`<Picture 1> ... is fully referenced`), blank line, phir wahi 3 fields
**Ref2VA** — chhe sections: `subject_definitions` · `summary` · `retention_analysis` · `detailed_description` · `overall_soundscape` · `non_diegetic_music`

Rules: pehla shot bina timestamp · baqi `[Shot 2] At 00:04.500, the camera cuts to …` · camera = type+amplitude+speed ·
dialogue `<d>[Language] exact words</d>` · speaker ID `(S1)` `</d>` ke bahar.

⚠️ H3 ke stable 11 languages: Arabic, Chinese, English, French, German, Italian, Japanese, Korean, Portuguese,
Russian, Spanish — **Urdu/Hindi stable list me nahi**. Pehle ek 5 s test clip par line check karo.

In [ ]:
#@title 4 · Monarch prompt templates + build_prompt()
TEMPLATES = {}

TEMPLATES["t2va.cinematic"] = """integrated_multimodal_description: [Shot 1] {style}, {shot}, {subject}. {camera}. {action}

overall_soundscape: {soundscape}

non_diegetic_music: {music}"""

TEMPLATES["i2va.first_frame"] = """For the target video, at 0.00 seconds into the target video, <Picture 1> (from [Shot 1]) is fully referenced.

integrated_multimodal_description: [Shot 1] {style}, beginning exactly from <Picture 1>: {subject}. {action}. {camera}

overall_soundscape: {soundscape}

non_diegetic_music: {music}"""

TEMPLATES["fl2va.first_last"] = """How the reference pictures align with the target video — Picture 1 (from Shot 1) aligns with the 0.00-second mark of the target video; Picture 2 (from Shot 1) aligns with the {end_time}-second mark of the target video.

integrated_multimodal_description: [Shot 1] {style}, starting exactly from Picture 1 and landing exactly on Picture 2 at {end_time} seconds. {subject}. {action}. {camera}

overall_soundscape: {soundscape}

non_diegetic_music: {music}"""

MONARCH_SUBJECT = (
    "<Subject 1> is the man in <Picture 1>–<Picture 5>: late-20s South Asian man, thick swept-back black hair "
    "with volume at the crown, long side locks in front of the ears, light beard along the jaw, straight nose with "
    "a slightly dropped tip, thin lips set back, wearing a fitted black shirt with the top button open and a "
    "charcoal blazer. <Picture 1> front hero, <Picture 2> three-quarter, <Picture 3> right profile, "
    "<Picture 4> warm smile reference, <Picture 5> intense gaze reference. "
    "<Audio 1> is the voice-timbre reference for <Subject 1>."
)

TEMPLATES["ref2va.monarch"] = """subject_definitions:
{subjects}

summary:
[single character, direct address] A {duration}-second {ratio} shot of <Subject 1> in {environment}, lit by a soft
key from upper-left with teal and amber rim lights, performing {beat}.

retention_analysis:
<Subject 1>: fully_preserved - same face, hair volume, side locks, beard line, nose and wardrobe as the reference
pictures; only expression, head angle and hand gesture change.
<Audio 1>: reference - the target speech follows the timbre and calm pacing of <Audio 1>.

detailed_description:
The target video is in realistic photographic style.
[Shot 1] {description}

overall_soundscape:
{soundscape}

non_diegetic_music:
{music}"""

def build_prompt(mode, **kw):
    tpl = TEMPLATES[mode]
    fields = dict(style="Cinematic, live-action", shot="a medium close-up frames",
                  subject="", camera="The camera holds a static shot", action="",
                  soundscape="quiet room tone", music="none", end_time="5.00",
                  subjects=MONARCH_SUBJECT, duration=5, ratio="9:16",
                  environment="a charcoal studio", beat="one direct-to-camera line",
                  description="")
    fields.update(kw)
    return tpl.format(**fields)

# quick sanity print of the Monarch Ref2VA skeleton
print(build_prompt("ref2va.monarch",
                   description="A medium close-up frames <Subject 1> from the chest up against a charcoal backdrop. "
                               "He holds eye contact with the lens and says with a low, assured voice (S1), "
                               "<d>[English] I built this studio in one night.</d> "
                               "The camera pushes in with small amplitude at slow speed.",
                   soundscape="A quiet studio room tone with a very low air-conditioning hum.",
                   music="A sparse detuned 808 sub under a muted cello note, fading to silence.")[:700], "…")

In [ ]:
#@title 5 · API helpers — create / poll / download / run_shot / cost
def data_url(path):
    """Local file -> base64 data URL. Images <=30MB, audio <=15MB. Prefer a public URL for anything bigger."""
    p = pathlib.Path(path)
    mime = mimetypes.guess_type(p.name)[0] or "application/octet-stream"
    return f"data:{mime};base64," + base64.b64encode(p.read_bytes()).decode()

def as_url(value):
    """Pass through http(s) URLs, convert local paths to data URLs."""
    return value if str(value).startswith(("http://", "https://", "data:")) else data_url(value)

def md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def create_task(payload):
    r = requests.post(CREATE, headers=HEADERS, json=payload, timeout=120)
    if r.status_code != 200:
        raise RuntimeError(f"create failed {r.status_code}: {r.text[:600]}")
    data = r.json()
    if "task_id" not in data:
        raise RuntimeError(f"no task_id in response: {data}")
    return data["task_id"]

def query_task(task_id):
    r = requests.get(QUERY.format(task_id=task_id), headers=HEADERS, timeout=60)
    r.raise_for_status()
    return r.json().get("task", {})

def poll(task_id, timeout=1800, interval=15, verbose=True):
    t0 = time.time()
    while time.time() - t0 < timeout:
        task = query_task(task_id)
        status = task.get("status")
        if verbose:
            print(f"  [{int(time.time()-t0):4d}s] {status}")
        if status == "succeeded":
            return task
        if status in ("failed", "cancelled"):
            raise RuntimeError(f"task {task_id} {status}: {json.dumps(task)[:600]}")
        time.sleep(interval)
    raise TimeoutError(f"task {task_id} still running after {timeout}s — task_id note kar lo, paisa already laga hai")

def estimate_cost(shots):
    total, rows = 0.0, []
    for s in shots:
        model, res, dur = s["model"], s["resolution"], int(s["duration"])
        rate = RATES.get(model, {}).get(res)
        if rate is None:
            rows.append([s["id"], model, res, dur, "?"]); continue
        extra_imgs = max(0, len(s.get("references", [])) - 5)
        ref_vid_s = s.get("reference_video_seconds", 0)
        cost = dur * rate + extra_imgs * 0.04 + ref_vid_s * rate
        total += cost
        rows.append([s["id"], model, res, dur, f"${cost:.2f}"])
    print(f"{'shot':<12}{'model':<24}{'res':<6}{'sec':>4}  cost")
    for rid, model, res, dur, c in rows:
        print(f"{rid:<12}{model:<24}{res:<6}{dur:>4}  {c}")
    print(f"\nESTIMATE: ${total:.2f}  (+ retry buffer 1.5× = ${total*1.5:.2f})")
    return total

def run_shot(spec, out_dir=OUT_DIR, do_download=True, timeout=1800):
    content = [{"type": "text", "text": spec["prompt"]}]

    if spec.get("first_frame"):
        content.append({"type": "image_url", "image_url": {"url": as_url(spec["first_frame"])},
                        "role": "first_frame"})
    if spec.get("last_frame"):
        content.append({"type": "image_url", "image_url": {"url": as_url(spec["last_frame"])},
                        "role": "last_frame"})
    for ref in spec.get("references", []):
        content.append({"type": "image_url", "image_url": {"url": as_url(ref)}, "role": "reference_image"})
    for vid in spec.get("reference_videos", []):
        content.append({"type": "video_url", "video_url": {"url": as_url(vid)}, "role": "reference_video"})
    for aud in spec.get("reference_audio", []):
        content.append({"type": "audio_url", "audio_url": {"url": as_url(aud)}, "role": "reference_audio"})

    payload = {"model": spec["model"], "content": content,
               "resolution": spec["resolution"], "duration": int(spec["duration"])}
    if spec.get("ratio"):
        payload["ratio"] = spec["ratio"]

    print(f"\n▶ {spec['id']} · {spec['model']} · {spec['resolution']} · {spec['duration']}s")
    task_id = create_task(payload)
    print("  task_id:", task_id)
    task = poll(task_id, timeout=timeout)

    receipt = {"id": spec["id"], "task_id": task_id, "model": payload["model"],
               "resolution": payload["resolution"], "duration": payload["duration"],
               "ratio": task.get("ratio"), "usage": task.get("usage", {}),
               "url": task.get("content", {}).get("url"), "created_at": int(time.time())}

    if do_download and receipt["url"]:
        dest = out_dir / f"{spec['id']}.mp4"
        with requests.get(receipt["url"], stream=True, timeout=300) as r:
            r.raise_for_status()
            with open(dest, "wb") as f:
                for chunk in r.iter_content(1 << 20):
                    f.write(chunk)
        receipt["local_path"] = str(dest)
        receipt["md5"] = md5(dest)
        receipt["bytes"] = dest.stat().st_size
        print(f"  ✔ {dest}  ({receipt['bytes']/1e6:.1f} MB, md5 {receipt['md5'][:10]}…)")
    return receipt

In [ ]:
#@title 6 · Output folder (optional: Drive)
USE_DRIVE = True   #@param {type:"boolean"}
DRIVE_SUBDIR = "MONARCH STUDIO/WF-06 · MiniMax H3"   #@param {type:"string"}

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT_DIR = pathlib.Path("/content/drive/MyDrive") / DRIVE_SUBDIR / "shots" / RUN_ID
        OUT_DIR.mkdir(parents=True, exist_ok=True)
    except Exception as e:
        print("Drive mount skip:", e)
print("Output dir:", OUT_DIR)

## 7 · Shot list — yahan kaam likho

Har shot: `id`, `model`, `resolution`, `duration`, `ratio`, `prompt`, aur optional `references` (ref2va),
`reference_audio` (voice timbre), `first_frame` / `last_frame` (i2va / fl2va).

Ratios: `16:9` `21:9` `9:16` `4:3` `3:4` `1:1` · t2va me `adaptive` allowed nahi · i2va me auto `adaptive`.

Reference images: `styles/minimax-h3-video/assets/refs/` ya Drive se local path. **Same 3–4 images har shot me.**

In [ ]:
#@title 8 · Shot list (edit me) — 3 ready examples
REFS = [
    # "assets/refs/MONARCH_REAL_FRONT.png",
    # "assets/refs/MONARCH_REAL_34.png",
    # "assets/refs/MONARCH_REAL_PROFILE.png",
]
VOICE_REF = []   # e.g. ["assets/refs/monarch_voice_ref.mp3"]

SHOTS = [
    # --- L1 · identity + voice test (Ref2VA, 5 s, vertical) ---
    {
        "id": "shot-01-monarch-talk",
        "model": "MiniMax-H3",
        "resolution": "768P",       # pehle sasta test; approve hone par "2K"
        "duration": 5,
        "ratio": "9:16",
        "prompt": build_prompt(
            "ref2va.monarch", duration=5, ratio="9:16", environment="a charcoal studio",
            beat="one direct-to-camera line",
            description=("A medium close-up frames <Subject 1> from the chest up, centred against a charcoal "
                         "backdrop with a faint teal rim on his left shoulder and a warm amber edge on his right. "
                         "Soft key light from upper-left models the cheekbone and the jaw beard. He holds eye "
                         "contact with the lens, inhales once, and speaks with a low, assured voice (S1), "
                         "<d>[English] I built this studio in one night.</d> As he finishes, the corner of his "
                         "mouth lifts into a slow confident smirk and his chin lowers slightly. The camera pushes "
                         "in with small amplitude at slow speed throughout the shot."),
            soundscape=("A quiet studio room tone with a very low air-conditioning hum; the faint rustle of the "
                        "blazer collar as his shoulders settle after the line."),
            music=("A sparse, modern trap-orchestral bed at slow tempo: a single detuned 808 sub under a muted "
                   "cello note, no drums, fading to silence on the smirk."),
        ),
        "references": REFS,
        "reference_audio": VOICE_REF,
    },

    # --- B-roll · T2VA cinematic plate (16:9, 8 s) ---
    {
        "id": "shot-02-noir-plate",
        "model": "MiniMax-H3",
        "resolution": "768P",
        "duration": 8,
        "ratio": "16:9",
        "prompt": build_prompt(
            "t2va.cinematic",
            style="Cinematic, live-action noir",
            shot="a low-angle wide shot frames a rain-wet street at 3 a.m.",
            subject=("A narrow cobblestone alley between shuttered shopfronts, one flickering neon sign in "
                     "amber, an overflowing storm drain steaming at the kerb"),
            action=("A lone figure in a long charcoal coat walks away from the camera, their shadow stretching "
                    "ahead of them along the wet stones"),
            camera="The camera tracks the figure with small amplitude at slow speed",
            soundscape=("Steady rain on metal awnings, a distant tram bell, footsteps in shallow water, a low "
                        "electrical buzz from the neon tube"),
            music="Solo muted trumpet over a sustained low string drone, slow tempo, ending unresolved",
        ),
    },

    # --- Reel · 9:16 product-style insert (5 s) ---
    {
        "id": "shot-03-reel-insert",
        "model": "MiniMax-H3-Lite",   # sasta explore pass ($0.04/s)
        "resolution": "480P",
        "duration": 5,
        "ratio": "9:16",
        "prompt": build_prompt(
            "t2va.cinematic",
            style="3D CG render",
            shot="a macro product shot frames a matte black object rotating on a reflective obsidian plinth",
            subject="single hard key light sweeping across its machined edge, dust motes drifting in the beam",
            action=("The object locks into place with a soft click and a teal gradient blooms behind it, leaving "
                    "the lower third of the frame empty dark space for typography"),
            camera="The camera arcs around the object with small amplitude at slow speed",
            soundscape="A deep sub-bass swell, a metallic click, subtle room air",
            music="Minimal electronic pulse at 90 BPM, one filtered saw chord that opens up at the cut",
        ),
    },
]

print(f"{len(SHOTS)} shot(s) ready:", ", ".join(s["id"] for s in SHOTS))

In [ ]:
#@title 9 · COST GATE — pehle estimate dekho, phir spend unlock karo
estimate_cost(SHOTS)

DRY_RUN        = True    #@param {type:"boolean"}   <- False karne se pehle estimate parh lo
CONFIRM_SPEND  = False   #@param {type:"boolean"}   <- True = "haan, paisa lagao"

if not DRY_RUN and not CONFIRM_SPEND:
    print("\n⛔ Spending locked. DRY_RUN=False karo AUR CONFIRM_SPEND=True karo.")

In [ ]:
#@title 10 · Run the batch
RECEIPTS = []
if not DRY_RUN and CONFIRM_SPEND:
    for spec in SHOTS:
        try:
            RECEIPTS.append(run_shot(spec))
        except Exception as e:
            print(f"✖ {spec['id']}: {e}")
            RECEIPTS.append({"id": spec["id"], "error": str(e)})
        (OUT_DIR / "receipts.json").write_text(json.dumps(RECEIPTS, indent=2))
else:
    print("Dry run — kuch submit nahi hua. Receipts:", OUT_DIR / "receipts.json")

print("\nActual spend (billed output seconds):")
spent = sum(r.get("usage", {}).get("output_seconds", 0) *
            RATES.get(r.get("model"), {}).get(r.get("resolution"), 0)
            for r in RECEIPTS if "usage" in r)
print(f"  ${spent:.2f}")

In [ ]:
#@title 11 · Status / retry (task_id se dobara — paisa dobara nahi lagta)
def retry_from_receipt(receipt):
    """Re-query + download a task that already exists. Safe: no new charge."""
    if not receipt.get("task_id"):
        raise ValueError("no task_id")
    task = poll(receipt["task_id"])
    url = task.get("content", {}).get("url")
    if not url:
        raise RuntimeError("no url in task")
    dest = OUT_DIR / f"{receipt['id']}.mp4"
    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        dest.write_bytes(r.content)
    receipt.update({"url": url, "local_path": str(dest), "md5": md5(dest), "usage": task.get("usage", {})})
    print("✔ recovered", dest)
    return receipt

# for r in RECEIPTS:
#     if r.get("error"):
#         retry_from_receipt(r)

In [ ]:
#@title 12 · Assemble to one master (ffmpeg)
ASSEMBLE   = False   #@param {type:"boolean"}
MASTER_FPS = 24      #@param {type:"integer"}

if ASSEMBLE:
    clips = sorted(p for p in OUT_DIR.glob("shot-*.mp4"))
    if not clips:
        print("no clips in", OUT_DIR)
    else:
        lst = OUT_DIR / "concat.txt"
        lst.write_text("".join(f"file '{p}'\n" for p in clips))
        master = OUT_DIR / "MONARCH_H3_MASTER.mp4"
        subprocess.run(["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", str(lst),
                        "-r", str(MASTER_FPS), "-c:v", "libx264", "-crf", "18", "-pix_fmt", "yuv420p",
                        "-c:a", "aac", "-b:a", "192k", str(master)], check=True)
        print("✔", master, round(master.stat().st_size / 1e6, 1), "MB")
        print(subprocess.run(["ffprobe", "-v", "error", "-show_entries",
                              "stream=codec_type,width,height,r_frame_rate,duration",
                              "-of", "default=nw=1", str(master)],
                             capture_output=True, text=True).stdout)

## 13 · Route B — local weights (sach, bina umeed)

Yeh notebook jaan-boojh kar **local H3-Base nahi** chalata. Wajah numbers hain, susti nahi:

- H3-Base resident BF16 ≈ **100–128 GB** (33B transformer + Qwen3-VL-32B encoder + VAEs). Colab free = 16 GB, Pro = 40 GB.
- Colab Pro A100 80GB par sirf **pruned INT8 (~42 GB)** + layer-offload se 768P mumkin hai, ~15–40 min per 5 s clip,
  aur 90-min idle disconnect par sab khatam. **2K local se mumkin nahi** (Regenerate-2K closed-source hai).
- Colab compute ko long-lived model server banana TOS ke khilaf hai.

Agar phir bhi local chahiye: apna GPU box ya hourly GPU rent (RunPod/Vast) + ComfyUI official template:

```bash
# ComfyUI route (local 24GB+ card, pruned INT8)
python main.py --lowvram        # + video_minimax_h3_r2v.json / video_minimax_h3_t2v.json templates

# SGLang route (official, 4xH100 class)
sglang serve --model-path MiniMaxAI/MiniMax-H3 --num-gpus 4 --ulysses-degree 4 \
  --performance-mode speed --host 0.0.0.0 --port 30010 --model-variant fl2va
```

Details: `knowledge/01_COLAB_RUNBOOK.md` §4 aur `knowledge/03_ROUTES_AND_COST.md`.